In [1]:
"""
Local Plotting Script for Weak-Measurement Expressibility Scaling Study.
Reads `expressibility_weak_s_results.pkl` and plots:
  1. Expressibility epsilon_U^(rho) vs p across N (log and linear scale)
  2. Frame Potential F_U^(rho) vs p across N (with dashed Haar bounds)
These are saved as separate figures.
"""

import os
import argparse
import pickle
import numpy as np
import matplotlib.pyplot as plt


def plot_expressibility_results(pkl_path, out_dir="plots_expressibility", log_scale=True):
    if not os.path.exists(pkl_path):
        raise FileNotFoundError(f"Could not find pickle file at: {pkl_path}")

    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    os.makedirs(out_dir, exist_ok=True)

    N_list = sorted(data.get("N_list", []))
    s_list = sorted(data.get("s_list", []), reverse=True)
    ansatz_list = data.get("ansatz_list", ["HEA1", "HEA2", "HVA"])
    depth_desc = data.get("depth_desc", "")
    samples = data.get("samples", 0)

    expressibilities = data.get("expressibilities", {})
    frame_potentials = data.get("frame_potentials", {})
    haar_potentials = data.get("haar_frame_potentials", {})

    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_list)))
    markers = ["o", "s", "^", "D", "v", "p", "*"]

    for s_val in s_list:
        if s_val not in expressibilities:
            continue

        active_ansatzes = [
            a for a in ansatz_list
            if a in expressibilities[s_val] and len(expressibilities[s_val][a]) > 0
        ]
        if not active_ansatzes:
            continue

        n_cols = len(active_ansatzes)
        
        # Create separate figures for Expressibility and Frame Potential
        fig_eps, axes_eps = plt.subplots(1, n_cols, figsize=(5.5 * n_cols, 5), squeeze=False)
        fig_fp, axes_fp = plt.subplots(1, n_cols, figsize=(5.5 * n_cols, 5), squeeze=False)

        for col_idx, ansatz in enumerate(active_ansatzes):
            ax_eps = axes_eps[0, col_idx]
            ax_fp = axes_fp[0, col_idx]

            p_dict_eps = expressibilities[s_val][ansatz]
            p_dict_fp = frame_potentials[s_val][ansatz]
            p_vals_all = sorted(p_dict_eps.keys())

            for n_idx, N in enumerate(N_list):
                p_valid, eps_valid, fp_valid = [], [], []
                for p in p_vals_all:
                    if N in p_dict_eps[p]:
                        p_valid.append(p)
                        eps_valid.append(p_dict_eps[p][N])
                        fp_valid.append(p_dict_fp[p][N])

                if not p_valid:
                    continue

                color = cmap[n_idx]
                marker = markers[n_idx % len(markers)]

                # Plot 1: Expressibility epsilon_U^(rho)
                eps_arr = np.array(eps_valid, dtype=float)
                if log_scale:
                    # Mask exact zeros from clamping if using log scale
                    eps_plot = np.where(eps_arr > 0, eps_arr, np.nan)
                else:
                    eps_plot = eps_arr

                ax_eps.plot(
                    p_valid, eps_plot, marker=marker, color=color,
                    linewidth=1.8, markersize=5, label=f"$N = {N}$"
                )

                # Plot 2: Frame Potential F_U^(rho) + Haar floor
                ax_fp.plot(
                    p_valid, fp_valid, marker=marker, color=color,
                    linewidth=1.8, markersize=5, label=f"$N = {N}$"
                )
                if N in haar_potentials:
                    ax_fp.axhline(
                        haar_potentials[N], color=color, linestyle="--",
                        linewidth=1.0, alpha=0.55
                    )

            # Format Expressibility Axes
            ax_eps.set_title(f"{ansatz} ($s = {s_val:.2f}$, {depth_desc})", fontsize=12, fontweight="bold")
            ax_eps.set_xlabel("Measurement Rate $p$", fontsize=11)
            ax_eps.set_ylabel(r"Expressibility $\varepsilon_{\mathbb{U}}^{\rho}$", fontsize=11)
            if log_scale:
                ax_eps.set_yscale("log")
            ax_eps.grid(True, which="both", linestyle=":", alpha=0.6)
            ax_eps.legend(fontsize=9, loc="best")

            # Format Frame Potential Axes
            ax_fp.set_title(f"{ansatz} Frame Potential vs Haar Limit", fontsize=11)
            ax_fp.set_xlabel("Measurement Rate $p$", fontsize=11)
            ax_fp.set_ylabel(r"Frame Potential $\mathcal{F}_{\mathbb{U}}^{(\rho)}$", fontsize=11)
            ax_fp.set_yscale("log")
            ax_fp.grid(True, which="both", linestyle=":", alpha=0.6)
            ax_fp.legend(fontsize=9, loc="best")

        # Save Expressibility Figure
        fig_eps.suptitle(
            f"Weak-Measurement Expressibility Scaling ($s = {s_val:.2f}$, $K = {samples}$ Trajectories)",
            fontsize=14, fontweight="bold", y=0.98
        )
        fig_eps.tight_layout(rect=[0, 0, 1, 0.95])
        scale_tag = "log" if log_scale else "linear"
        out_file_eps = os.path.join(out_dir, f"expressibility_eps_s{s_val:.2f}_{scale_tag}.png")
        fig_eps.savefig(out_file_eps, dpi=300)
        plt.close(fig_eps)
        print(f"Saved Expressibility figure -> {out_file_eps}")

        # Save Frame Potential Figure
        fig_fp.suptitle(
            f"Weak-Measurement Frame Potential ($s = {s_val:.2f}$, $K = {samples}$ Trajectories)",
            fontsize=14, fontweight="bold", y=0.98
        )
        fig_fp.tight_layout(rect=[0, 0, 1, 0.95])
        out_file_fp = os.path.join(out_dir, f"frame_potential_s{s_val:.2f}.png")
        fig_fp.savefig(out_file_fp, dpi=300)
        plt.close(fig_fp)
        print(f"Saved Frame Potential figure -> {out_file_fp}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--pkl", type=str,
        default="expressibility_weak_s_results1.pkl",
        help="Path to expressibility_weak_s_results.pkl"
    )
    parser.add_argument("--out-dir", type=str, default="plots_expressibility")
    parser.add_argument("--linear", action="store_true", help="Plot expressibility on a linear y-axis instead of log")
    args, unknown = parser.parse_known_args()
    plot_expressibility_results(args.pkl, out_dir=args.out_dir, log_scale=(not args.linear))

Saved Expressibility figure -> plots_expressibility\expressibility_eps_s1.00_log.png
Saved Frame Potential figure -> plots_expressibility\frame_potential_s1.00.png
Saved Expressibility figure -> plots_expressibility\expressibility_eps_s0.50_log.png
Saved Frame Potential figure -> plots_expressibility\frame_potential_s0.50.png


In [4]:
import os
import argparse
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D  # Added for custom legend

def plot_expressibility_comparison(pkl_path, target_s=[1.0, 0.5], out_dir="plots_expressibility_compare", log_scale=True):
    if not os.path.exists(pkl_path):
        raise FileNotFoundError(f"Could not find pickle file at: {pkl_path}")

    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    os.makedirs(out_dir, exist_ok=True)

    N_list = sorted(data.get("N_list", []))
    ansatz_list = data.get("ansatz_list", ["HEA1", "HEA2", "HVA"])
    depth_desc = data.get("depth_desc", "")
    samples = data.get("samples", 0)

    expressibilities = data.get("expressibilities", {})
    frame_potentials = data.get("frame_potentials", {})
    haar_potentials = data.get("haar_frame_potentials", {})

    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_list)))
    markers = ["o", "s", "^", "D", "v", "p", "*"]
    linestyles = ["-", "--", ":", "-."]

    active_ansatzes = set()
    for s_val in target_s:
        if s_val in expressibilities:
            active_ansatzes.update([a for a in ansatz_list if a in expressibilities[s_val] and len(expressibilities[s_val][a]) > 0])
    active_ansatzes = sorted(list(active_ansatzes))

    if not active_ansatzes:
        print("No data found for the specified target_s values.")
        return

    n_cols = len(active_ansatzes)
    
    fig_eps, axes_eps = plt.subplots(1, n_cols, figsize=(6.5 * n_cols, 5.5), squeeze=False)
    fig_fp, axes_fp = plt.subplots(1, n_cols, figsize=(6.5 * n_cols, 5.5), squeeze=False)

    # Create custom legend elements once
    custom_legend = []
    # 1. Elements for N (Colors & Markers)
    for n_idx, N in enumerate(N_list):
        custom_legend.append(Line2D([0], [0], color=cmap[n_idx], marker=markers[n_idx % len(markers)], 
                                    linestyle='-', linewidth=1.8, markersize=5, label=f'$N = {N}$'))
    # 2. Add a blank spacer for neatness
    custom_legend.append(Line2D([0], [0], color='none', label=''))
    # 3. Elements for s (Line styles)
    for s_idx, s_val in enumerate(target_s):
        custom_legend.append(Line2D([0], [0], color='black', linestyle=linestyles[s_idx % len(linestyles)], 
                                    linewidth=1.8, label=f'$s = {s_val}$'))

    for col_idx, ansatz in enumerate(active_ansatzes):
        ax_eps = axes_eps[0, col_idx]
        ax_fp = axes_fp[0, col_idx]

        for s_idx, s_val in enumerate(target_s):
            if s_val not in expressibilities or ansatz not in expressibilities[s_val]:
                continue
                
            ls = linestyles[s_idx % len(linestyles)]
            p_dict_eps = expressibilities[s_val][ansatz]
            p_dict_fp = frame_potentials[s_val][ansatz]
            p_vals_all = sorted(p_dict_eps.keys())

            for n_idx, N in enumerate(N_list):
                p_valid, eps_valid, fp_valid = [], [], []
                for p in p_vals_all:
                    if N in p_dict_eps[p]:
                        p_valid.append(p)
                        eps_valid.append(p_dict_eps[p][N])
                        fp_valid.append(p_dict_fp[p][N])

                if not p_valid:
                    continue

                color = cmap[n_idx]
                marker = markers[n_idx % len(markers)]

                # Note: We omit the 'label' argument here since we use the custom legend
                eps_arr = np.array(eps_valid, dtype=float)
                if log_scale:
                    eps_plot = np.where(eps_arr > 0, eps_arr, np.nan)
                else:
                    eps_plot = eps_arr

                ax_eps.plot(
                    p_valid, eps_plot, marker=marker, color=color, linestyle=ls,
                    linewidth=1.8, markersize=5
                )

                ax_fp.plot(
                    p_valid, fp_valid, marker=marker, color=color, linestyle=ls,
                    linewidth=1.8, markersize=5
                )
                
                if s_idx == 0 and N in haar_potentials:
                    ax_fp.axhline(
                        haar_potentials[N], color=color, linestyle=":", 
                        linewidth=1.5, alpha=0.4
                    )

        # Apply the custom legend to both plots
        ax_eps.set_title(f"{ansatz} ({depth_desc})", fontsize=12, fontweight="bold")
        ax_eps.set_xlabel("Measurement Rate $p$", fontsize=11)
        ax_eps.set_ylabel(r"Expressibility $\varepsilon_{\mathbb{U}}^{\rho}$", fontsize=11)
        if log_scale:
            ax_eps.set_yscale("log")
        ax_eps.grid(True, which="both", linestyle=":", alpha=0.6)
        ax_eps.legend(handles=custom_legend, fontsize=9, loc="best", ncol=2) 

        ax_fp.set_title(f"{ansatz} Frame Potential", fontsize=12, fontweight="bold")
        ax_fp.set_xlabel("Measurement Rate $p$", fontsize=11)
        ax_fp.set_ylabel(r"Frame Potential $\mathcal{F}_{\mathbb{U}}^{(\rho)}$", fontsize=11)
        ax_fp.set_yscale("log")
        ax_fp.grid(True, which="both", linestyle=":", alpha=0.6)
        ax_fp.legend(handles=custom_legend, fontsize=9, loc="best", ncol=2)

    s_label = "_".join([str(s) for s in target_s])
    
    fig_eps.suptitle(
        f"Expressibility Comparison ($s \in {target_s}$, $K = {samples}$)",
        fontsize=14, fontweight="bold", y=0.98
    )
    fig_eps.tight_layout(rect=[0, 0, 1, 0.95])
    scale_tag = "log" if log_scale else "linear"
    out_file_eps = os.path.join(out_dir, f"expressibility_comp_s{s_label}_{scale_tag}.png")
    fig_eps.savefig(out_file_eps, dpi=300)
    plt.close(fig_eps)
    print(f"Saved Expressibility comparison -> {out_file_eps}")

    fig_fp.suptitle(
        f"Frame Potential Comparison ($s \in {target_s}$, $K = {samples}$)",
        fontsize=14, fontweight="bold", y=0.98
    )
    fig_fp.tight_layout(rect=[0, 0, 1, 0.95])
    out_file_fp = os.path.join(out_dir, f"frame_potential_comp_s{s_label}.png")
    fig_fp.savefig(out_file_fp, dpi=300)
    plt.close(fig_fp)
    print(f"Saved Frame Potential comparison -> {out_file_fp}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--pkl", type=str, default="expressibility_weak_s_results1.pkl")
    parser.add_argument("--out-dir", type=str, default="plots_expressibility_compare")
    parser.add_argument("--linear", action="store_true")
    parser.add_argument("--s-vals", nargs='+', type=float, default=[1.0, 0.5], help="List of s values to compare")
    args, unknown = parser.parse_known_args()
    
    plot_expressibility_comparison(args.pkl, target_s=args.s_vals, out_dir=args.out_dir, log_scale=(not args.linear))

<>:127: SyntaxWarning: invalid escape sequence '\i'
<>:138: SyntaxWarning: invalid escape sequence '\i'
<>:127: SyntaxWarning: invalid escape sequence '\i'
<>:138: SyntaxWarning: invalid escape sequence '\i'
C:\Users\nimma\AppData\Local\Temp\ipykernel_37880\1536040707.py:127: SyntaxWarning: invalid escape sequence '\i'
  f"Expressibility Comparison ($s \in {target_s}$, $K = {samples}$)",
C:\Users\nimma\AppData\Local\Temp\ipykernel_37880\1536040707.py:138: SyntaxWarning: invalid escape sequence '\i'
  f"Frame Potential Comparison ($s \in {target_s}$, $K = {samples}$)",


Saved Expressibility comparison -> plots_expressibility_compare\expressibility_comp_s1.0_0.5_log.png
Saved Frame Potential comparison -> plots_expressibility_compare\frame_potential_comp_s1.0_0.5.png


In [4]:
import os
import argparse
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def plot_expressibility_comparison(pkl_path, target_s=[1.0, 0.5], out_dir="plots_expressibility_compare", log_scale=True):
    if not os.path.exists(pkl_path):
        raise FileNotFoundError(f"Could not find pickle file at: {pkl_path}")

    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    os.makedirs(out_dir, exist_ok=True)

    N_list = sorted(data.get("N_list", []))
    ansatz_list = data.get("ansatz_list", ["HEA1", "HEA2", "HVA"])
    depth_desc = data.get("depth_desc", "")
    samples = data.get("samples", 0)

    expressibilities = data.get("expressibilities", {})
    frame_potentials = data.get("frame_potentials", {})
    haar_potentials = data.get("haar_frame_potentials", {})

    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_list)))
    markers = ["o", "s", "^", "D", "v", "p", "*"]
    linestyles = ["-", "--", ":", "-."]

    active_ansatzes = set()
    for s_val in target_s:
        if s_val in expressibilities:
            active_ansatzes.update([a for a in ansatz_list if a in expressibilities[s_val] and len(expressibilities[s_val][a]) > 0])
    active_ansatzes = sorted(list(active_ansatzes))

    if not active_ansatzes:
        print("No data found for the specified target_s values.")
        return

    # Create custom legend elements once
    custom_legend = []
    # 1. Elements for N (Colors & Markers)
    for n_idx, N in enumerate(N_list):
        custom_legend.append(Line2D([0], [0], color=cmap[n_idx], marker=markers[n_idx % len(markers)], 
                                    linestyle='-', linewidth=1.8, markersize=5, label=f'$N = {N}$'))
    # 2. Add a blank spacer for neatness
    custom_legend.append(Line2D([0], [0], color='none', label=''))
    # 3. Elements for s (Line styles)
    for s_idx, s_val in enumerate(target_s):
        custom_legend.append(Line2D([0], [0], color='black', linestyle=linestyles[s_idx % len(linestyles)], 
                                    linewidth=1.8, label=f'$s = {s_val}$'))

    s_label = "_".join([str(s) for s in target_s])
    scale_tag = "log" if log_scale else "linear"

    # Iterate and create a separate plot for each ansatz
    for ansatz in active_ansatzes:
        fig_eps, ax_eps = plt.subplots(figsize=(7, 5.5))
        fig_fp, ax_fp = plt.subplots(figsize=(7, 5.5))

        for s_idx, s_val in enumerate(target_s):
            if s_val not in expressibilities or ansatz not in expressibilities[s_val]:
                continue
                
            ls = linestyles[s_idx % len(linestyles)]
            p_dict_eps = expressibilities[s_val][ansatz]
            p_dict_fp = frame_potentials[s_val][ansatz]
            p_vals_all = sorted(p_dict_eps.keys())

            for n_idx, N in enumerate(N_list):
                p_valid, eps_valid, fp_valid = [], [], []
                for p in p_vals_all:
                    if N in p_dict_eps[p]:
                        p_valid.append(p)
                        eps_valid.append(p_dict_eps[p][N])
                        fp_valid.append(p_dict_fp[p][N])

                if not p_valid:
                    continue

                color = cmap[n_idx]
                marker = markers[n_idx % len(markers)]

                eps_arr = np.array(eps_valid, dtype=float)
                if log_scale:
                    eps_plot = np.where(eps_arr > 0, eps_arr, np.nan)
                else:
                    eps_plot = eps_arr

                ax_eps.plot(
                    p_valid, eps_plot, marker=marker, color=color, linestyle=ls,
                    linewidth=1.8, markersize=5
                )

                ax_fp.plot(
                    p_valid, fp_valid, marker=marker, color=color, linestyle=ls,
                    linewidth=1.8, markersize=5
                )
                
                if s_idx == 0 and N in haar_potentials:
                    ax_fp.axhline(
                        haar_potentials[N], color=color, linestyle=":", 
                        linewidth=1.5, alpha=0.4
                    )

        # Style and format the Expressibility plot
        ax_eps.set_title(f"{ansatz} ({depth_desc})", fontsize=12, fontweight="bold")
        ax_eps.set_xlabel("Measurement Rate $p$", fontsize=11)
        ax_eps.set_ylabel(r"Expressibility $\varepsilon_{\mathbb{U}}^{\rho}$", fontsize=11)
        if log_scale:
            ax_eps.set_yscale("log")
        ax_eps.grid(True, which="both", linestyle=":", alpha=0.6)
        ax_eps.legend(handles=custom_legend, fontsize=9, loc="best", ncol=2) 

        fig_eps.suptitle(
            f"{ansatz} Expressibility ($s \in {target_s}$, $K = {samples}$)",
            fontsize=14, fontweight="bold", y=0.98
        )
        fig_eps.tight_layout(rect=[0, 0, 1, 0.95])
        
        out_file_eps = os.path.join(out_dir, f"{ansatz}_expressibility_s{s_label}_{scale_tag}.pdf")
        fig_eps.savefig(out_file_eps, dpi=300)
        plt.close(fig_eps)
        print(f"Saved {ansatz} Expressibility -> {out_file_eps}")

        # Style and format the Frame Potential plot
        ax_fp.set_title(f"{ansatz} Frame Potential", fontsize=12, fontweight="bold")
        ax_fp.set_xlabel("Measurement Rate $p$", fontsize=11)
        ax_fp.set_ylabel(r"Frame Potential $\mathcal{F}_{\mathbb{U}}^{(\rho)}$", fontsize=11)
        ax_fp.set_yscale("log")
        ax_fp.grid(True, which="both", linestyle=":", alpha=0.6)
        ax_fp.legend(handles=custom_legend, fontsize=9, loc="best", ncol=2)

        fig_fp.suptitle(
            f"{ansatz} Frame Potential ($s \in {target_s}$, $K = {samples}$)",
            fontsize=14, fontweight="bold", y=0.98
        )
        fig_fp.tight_layout(rect=[0, 0, 1, 0.95])
        
        out_file_fp = os.path.join(out_dir, f"{ansatz}_frame_potential_s{s_label}.pdf")
        fig_fp.savefig(out_file_fp, dpi=300)
        plt.close(fig_fp)
        print(f"Saved {ansatz} Frame Potential -> {out_file_fp}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--pkl", type=str, default="expressibility_weak_s_results1.pkl")
    parser.add_argument("--out-dir", type=str, default="plots_expressibility_compare")
    parser.add_argument("--linear", action="store_true")
    parser.add_argument("--s-vals", nargs='+', type=float, default=[1.0, 0.5], help="List of s values to compare")
    args, unknown = parser.parse_known_args()
    
    plot_expressibility_comparison(args.pkl, target_s=args.s_vals, out_dir=args.out_dir, log_scale=(not args.linear))

<>:116: SyntaxWarning: invalid escape sequence '\i'
<>:135: SyntaxWarning: invalid escape sequence '\i'
<>:116: SyntaxWarning: invalid escape sequence '\i'
<>:135: SyntaxWarning: invalid escape sequence '\i'
C:\Users\nimma\AppData\Local\Temp\ipykernel_21064\346571428.py:116: SyntaxWarning: invalid escape sequence '\i'
  f"{ansatz} Expressibility ($s \in {target_s}$, $K = {samples}$)",
C:\Users\nimma\AppData\Local\Temp\ipykernel_21064\346571428.py:135: SyntaxWarning: invalid escape sequence '\i'
  f"{ansatz} Frame Potential ($s \in {target_s}$, $K = {samples}$)",


Saved HEA1 Expressibility -> plots_expressibility_compare\HEA1_expressibility_s1.0_0.5_log.pdf
Saved HEA1 Frame Potential -> plots_expressibility_compare\HEA1_frame_potential_s1.0_0.5.pdf
Saved HEA2 Expressibility -> plots_expressibility_compare\HEA2_expressibility_s1.0_0.5_log.pdf
Saved HEA2 Frame Potential -> plots_expressibility_compare\HEA2_frame_potential_s1.0_0.5.pdf
Saved HVA Expressibility -> plots_expressibility_compare\HVA_expressibility_s1.0_0.5_log.pdf
Saved HVA Frame Potential -> plots_expressibility_compare\HVA_frame_potential_s1.0_0.5.pdf
